# BargainBall: data and model checks

One place to review data coverage, production-tier diagnostics, model performance, player comps, and comp-based salary references. This notebook reads prepared local outputs; it does **not** train models or load the test split.

Before running: run the data-preparation and model-training notebooks. Audit and comp-estimation implementations are included in this notebook. Generated `data/processed/`, `models/`, and `results/` are intentionally local and git-ignored.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

rootPath = Path.cwd()
while not (rootPath / 'data' / 'processed').exists() and rootPath != rootPath.parent:
    rootPath = rootPath.parent
if not (rootPath / 'data' / 'processed').exists():
    raise FileNotFoundError('Open this notebook from inside the BargainBall repository.')
processedPath = rootPath / 'data' / 'processed'
resultsPath = rootPath / 'results' / 'development' / 'multitask_profile'
modelsPath = rootPath / 'models' / 'numpy_multitask_profile'

def findPrimaryRun():
    runs = []
    for resultFolder in resultsPath.iterdir():
        metadataFile = modelsPath / resultFolder.name / 'metadata.json'
        if resultFolder.is_dir() and metadataFile.exists():
            modelMetadata = json.loads(metadataFile.read_text(encoding='utf-8'))
            if int(modelMetadata.get('training_minimum_prior_games', 0)) == 0:
                runs.append(resultFolder)
    if not runs:
        raise FileNotFoundError('No unrestricted profile-model run found. Run notebook 02 first.')
    return max(runs, key=lambda p: p.stat().st_mtime)

runFolder = findPrimaryRun()
runId = runFolder.name
print('Repository:', rootPath)
print('Primary unrestricted model run:', runId)
print('Test split will not be loaded.')
ROOT = rootPath
PROCESSED = processedPath
RESULTS_ROOT = resultsPath
MODEL_ROOT = modelsPath
SALARY_PATH = processedPath / 'salary_cap_normalized_pairs.csv'
QUANTILES = [0.20, 0.40, 0.60, 0.80]
TIER_NAMES = ['lowest', 'lower-middle', 'upper-middle', 'high', 'highest']
MIN_USABLE_COMPS = 5
QUERY_KEYS = ['query_player', 'query_player_id', 'query_prior_season', 'query_target_season']


## Implement the development-tier audit and comp-salary estimator

In [ ]:
def auditProfileTiers():
    result_dir = max(
        (path for path in RESULTS_ROOT.iterdir() if path.is_dir()),
        key=lambda path: path.stat().st_mtime,
    )
    train = pd.read_csv(PROCESSED / "train_rows_rates.csv")
    dev = pd.read_csv(PROCESSED / "dev_rows_rates.csv")
    predictions = pd.read_csv(result_dir / "development_profile_predictions.csv")
    if len(dev) != len(predictions):
        raise ValueError("Development metadata and predictions do not align.")
    if not dev[["player_id", "season", "target_season"]].equals(
        predictions[["player_id", "season", "target_season"]]
    ):
        raise ValueError("Development player-season order differs from predictions.")

    train_pie = train["pie"].to_numpy(dtype=float)
    if not np.isfinite(train_pie).all():
        raise ValueError("Training prior-season PIE must be complete for tiering.")
    boundaries = np.quantile(train_pie, QUANTILES)
    labels = np.searchsorted(boundaries, train_pie, side="right")
    train_distribution = []
    for tier_id, name in enumerate(TIER_NAMES):
        values = train_pie[labels == tier_id]
        train_distribution.append(
            {
                "prior_pie_tier": name,
                "training_rows": int(len(values)),
                "training_share": float(len(values) / len(train_pie)),
                "prior_pie_min": float(values.min()) if len(values) else None,
                "prior_pie_median": float(np.median(values)) if len(values) else None,
                "prior_pie_max": float(values.max()) if len(values) else None,
            }
        )

    dev_pie = dev["pie"].to_numpy(dtype=float)
    dev_tiers = np.searchsorted(boundaries, dev_pie, side="right")
    dev["prior_pie_tier"] = [TIER_NAMES[index] for index in dev_tiers]
    dev["prior_pie_training_percentile"] = np.searchsorted(
        np.sort(train_pie), dev_pie, side="right"
    ) / len(train_pie)
    predictions["prior_pie_tier"] = dev["prior_pie_tier"].to_numpy()
    predictions["prior_pie_training_percentile"] = (
        dev["prior_pie_training_percentile"].to_numpy()
    )

    target_names = [
        column.removeprefix("actual_")
        for column in predictions.columns
        if column.startswith("actual_")
        and "predicted_" + column.removeprefix("actual_") in predictions.columns
        and "prior_" + column.removeprefix("actual_") in predictions.columns
    ]
    by_tier = []
    for tier_name in TIER_NAMES:
        group = predictions[predictions["prior_pie_tier"] == tier_name]
        for target in target_names:
            actual = group["actual_" + target].to_numpy(dtype=float)
            predicted = group["predicted_" + target].to_numpy(dtype=float)
            baseline = group["prior_" + target].to_numpy(dtype=float)
            valid = np.isfinite(actual) & np.isfinite(predicted) & np.isfinite(baseline)
            if not valid.any():
                continue
            model_error = predicted[valid] - actual[valid]
            baseline_error = baseline[valid] - actual[valid]
            by_tier.append(
                {
                    "prior_pie_tier": tier_name,
                    "target": target,
                    "rows": int(valid.sum()),
                    "model_rmse": float(np.sqrt(np.mean(model_error**2))),
                    "model_mae": float(np.mean(np.abs(model_error))),
                    "persistence_rmse": float(np.sqrt(np.mean(baseline_error**2))),
                    "persistence_mae": float(np.mean(np.abs(baseline_error))),
                    "model_rmse_gain_pct": float(
                        (1 - np.sqrt(np.mean(model_error**2))
                         / np.sqrt(np.mean(baseline_error**2))) * 100
                    ),
                }
            )

    tails = []
    for target in target_names:
        actual = predictions["actual_" + target].to_numpy(dtype=float)
        predicted = predictions["predicted_" + target].to_numpy(dtype=float)
        baseline = predictions["prior_" + target].to_numpy(dtype=float)
        valid = np.isfinite(actual) & np.isfinite(predicted) & np.isfinite(baseline)
        model_squared = (predicted[valid] - actual[valid]) ** 2
        baseline_squared = (baseline[valid] - actual[valid]) ** 2
        n_tail = max(1, int(np.ceil(0.05 * len(model_squared))))
        model_order = np.argsort(model_squared)
        baseline_order = np.argsort(baseline_squared)
        tails.append(
            {
                "target": target,
                "rows": int(valid.sum()),
                "model_rmse": float(np.sqrt(np.mean(model_squared))),
                "model_rmse_without_largest_5pct_errors": float(
                    np.sqrt(np.mean(model_squared[model_order[:-n_tail]]))
                ),
                "model_top_5pct_squared_error_share": float(
                    model_squared[model_order[-n_tail:]].sum()
                    / model_squared.sum()
                ),
                "persistence_rmse": float(np.sqrt(np.mean(baseline_squared))),
                "persistence_top_5pct_squared_error_share": float(
                    baseline_squared[baseline_order[-n_tail:]].sum()
                    / baseline_squared.sum()
                ),
            }
        )

    distribution_frame = pd.DataFrame(train_distribution)
    tier_frame = pd.DataFrame(by_tier)
    tail_frame = pd.DataFrame(tails)
    distribution_frame.to_csv(result_dir / "training_prior_pie_tiers.csv", index=False)
    tier_frame.to_csv(result_dir / "development_metrics_by_prior_pie_tier.csv", index=False)
    tail_frame.to_csv(result_dir / "development_tail_error_audit.csv", index=False)

    report = {
        "tier_feature": "prior-season PIE",
        "tier_boundary_fit": "training split only",
        "training_quantile_boundaries": {
            str(q): float(value) for q, value in zip(QUANTILES, boundaries)
        },
        "training_tiers": train_distribution,
        "development_rows": int(len(dev)),
        "development_tier_counts": {
            name: int((dev["prior_pie_tier"] == name).sum()) for name in TIER_NAMES
        },
        "test_split_loaded": False,
        "model_retrained_or_reweighted": False,
        "files": [
            "training_prior_pie_tiers.csv",
            "development_metrics_by_prior_pie_tier.csv",
            "development_tail_error_audit.csv",
        ],
    }
    (result_dir / "prior_pie_tier_audit.json").write_text(
        json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )
    print("Results directory:", result_dir)
    print("\nTraining prior-PIE distribution:")
    print(distribution_frame.to_string(index=False))
    print("\nDevelopment metrics by tier (key outcomes):")
    key_metrics = tier_frame[tier_frame["target"].isin(["pie", "bpm", "ws_48", "minutes"])]
    print(key_metrics.to_string(index=False))
    print("\nExtreme-error sensitivity:")
    print(tail_frame.to_string(index=False))
    print("\nNo oversampling or reweighting performed; test split not loaded.")

In [ ]:
def find_latest_unrestricted_run():
    candidates = []
    for result_dir in RESULTS_ROOT.iterdir():
        metadata_path = MODEL_ROOT / result_dir.name / "metadata.json"
        comps_path = result_dir / "development_profile_neighbors.csv"
        if not metadata_path.exists() or not comps_path.exists():
            continue
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if int(metadata.get("training_minimum_prior_games", 0)) == 0:
            candidates.append(result_dir)
    if not candidates:
        raise FileNotFoundError("No unrestricted profile-model result run was found.")
    return max(candidates, key=lambda path: path.stat().st_mtime)

In [ ]:
def summarize_comps(comps, cap_by_target_season):
    rows = []
    for key, group in comps.groupby(QUERY_KEYS, sort=False, dropna=False):
        group = group.sort_values("comp_rank", kind="stable")
        share = pd.to_numeric(
            group["comp_target_salary_cap_share_pct"], errors="coerce"
        ).to_numpy(dtype=float)
        review = group["comp_salary_review_flag"].fillna(True).astype(bool).to_numpy()
        usable = np.isfinite(share) & (share >= 0) & ~review
        values = share[usable]
        valid_comps = group.loc[usable]
        target_season = int(key[3])
        cap = cap_by_target_season.get(target_season, np.nan)
        enough = len(values) >= MIN_USABLE_COMPS
        estimate_pct = float(np.median(values)) if enough else np.nan
        q25 = float(np.quantile(values, 0.25)) if enough else np.nan
        q75 = float(np.quantile(values, 0.75)) if enough else np.nan
        row = dict(zip(QUERY_KEYS, key))
        row.update(
            {
                "salary_estimate_method": (
                    "median salary-cap share of nearest historical profiles"
                ),
                "nearest_profile_comp_count": int(len(group)),
                "usable_salary_comp_count": int(len(values)),
                "minimum_usable_comps": MIN_USABLE_COMPS,
                "salary_estimate_available": bool(enough),
                "estimated_salary_cap_share_pct": estimate_pct,
                "salary_cap_share_q25_pct": q25,
                "salary_cap_share_q75_pct": q75,
                "target_season_salary_cap_usd": float(cap) if pd.notna(cap) else np.nan,
                "estimated_salary_usd": (
                    estimate_pct / 100 * float(cap)
                    if enough and pd.notna(cap)
                    else np.nan
                ),
                "salary_q25_usd": (
                    q25 / 100 * float(cap)
                    if enough and pd.notna(cap)
                    else np.nan
                ),
                "salary_q75_usd": (
                    q75 / 100 * float(cap)
                    if enough and pd.notna(cap)
                    else np.nan
                ),
                "usable_comps": "; ".join(
                    f"{comp.historical_comp} ({comp.comp_target_salary_cap_share_pct:.2f}%)"
                    for comp in valid_comps.itertuples()
                ),
                "estimate_warning": (
                    ""
                    if enough
                    else f"Only {len(values)} eligible comp salaries; fewer than {MIN_USABLE_COMPS}."
                ),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def estimateSalaryFromComps(run_id=None):
    result_dir = RESULTS_ROOT / run_id if run_id else find_latest_unrestricted_run()
    if not result_dir.is_dir():
        raise FileNotFoundError(f"Run directory not found: {result_dir}")
    metadata_path = MODEL_ROOT / result_dir.name / "metadata.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if int(metadata.get("training_minimum_prior_games", 0)) != 0:
        raise ValueError("Salary estimates should use the unrestricted primary model.")

    salary_data = pd.read_csv(SALARY_PATH)
    cap_table = salary_data[
        ["target_season", "salary_cap_usd"]
    ].dropna().drop_duplicates()
    cap_counts = cap_table.groupby("target_season")["salary_cap_usd"].nunique()
    if cap_counts.gt(1).any():
        raise ValueError("Salary-cap values are inconsistent within target season.")
    cap_by_target_season = cap_table.drop_duplicates("target_season").set_index(
        "target_season"
    )["salary_cap_usd"].to_dict()

    result_paths = [
        (
            "development_profile_neighbors.csv",
            "development_comp_salary_estimates.csv",
        ),
        (
            "development_bench_to_starter_neighbors.csv",
            "development_bench_to_starter_salary_estimates.csv",
        ),
    ]
    report = {
        "run_directory": str(result_dir.relative_to(ROOT)),
        "method": "Median salary-cap share among nearest profile comps with an available, non-flagged source salary.",
        "similarity_model": "Salary is not used in forecast training, PCA, distance calculation, or comp ranking.",
        "minimum_usable_comps": MIN_USABLE_COMPS,
        "test_split_loaded": False,
        "outputs": [],
    }
    for comps_file, output_file in result_paths:
        comps_path = result_dir / comps_file
        comps = pd.read_csv(comps_path)
        estimates = summarize_comps(comps, cap_by_target_season)
        output_path = result_dir / output_file
        estimates.to_csv(output_path, index=False)
        report["outputs"].append(
            {
                "file": output_file,
                "query_count": int(len(estimates)),
                "estimates_available": int(estimates["salary_estimate_available"].sum()),
                "estimates_insufficient_comps": int(
                    (~estimates["salary_estimate_available"]).sum()
                ),
            }
        )

    (result_dir / "comp_salary_estimate_report.json").write_text(
        json.dumps(report, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    all_estimates = pd.read_csv(result_dir / result_paths[0][1])
    bench_estimates = pd.read_csv(result_dir / result_paths[1][1])
    print("Primary model run:", result_dir)
    print("\nExample nearest-comp salary estimates:")
    print(
        all_estimates[
            [
                "query_player",
                "query_target_season",
                "usable_salary_comp_count",
                "estimated_salary_cap_share_pct",
                "salary_cap_share_q25_pct",
                "salary_cap_share_q75_pct",
                "estimated_salary_usd",
            ]
        ]
        .head(10)
        .to_string(index=False)
    )
    print(
        f"\nAll-player estimates available: "
        f"{all_estimates['salary_estimate_available'].sum()} / {len(all_estimates)}"
    )
    print(
        f"Bench-to-starter estimates available: "
        f"{bench_estimates['salary_estimate_available'].sum()} / {len(bench_estimates)}"
    )
    print("Salary is comp context only; it does not alter similarity rankings.")

## 1. Dataset coverage and sample-size checks
Review row counts, missing target labels, low-game samples, and extreme prior PIE values. The tier cutoffs below are descriptive; the model training data remain unchanged.

In [ ]:
trainRows = pd.read_csv(processedPath / 'train_rows_rates.csv')
devRows = pd.read_csv(processedPath / 'dev_rows_rates.csv')
playerPairs = pd.read_csv(processedPath / 'prediction_pairs_rates.csv')
dataSummary = pd.DataFrame([
    {'split':'train','rows':len(trainRows),'prior_games_lt_10':int((trainRows.g < 10).sum()),'prior_games_lt_20':int((trainRows.g < 20).sum()),'prior_PIE_min':trainRows.pie.min(),'prior_PIE_max':trainRows.pie.max()},
    {'split':'development','rows':len(devRows),'prior_games_lt_10':int((devRows.g < 10).sum()),'prior_games_lt_20':int((devRows.g < 20).sum()),'prior_PIE_min':devRows.pie.min(),'prior_PIE_max':devRows.pie.max()},
])
display(dataSummary)

extremePie = trainRows.loc[~trainRows.pie.between(-0.25, 0.40), ['player','season','g','mp','pie','ws_48','bpm']].sort_values('pie')
print('Prior-season PIE outside [-0.25, 0.40]: ' + str(len(extremePie)) + ' training rows')
display(extremePie)
print('Rows with prior-season games < 10:', int((trainRows.g < 10).sum()), '/', len(trainRows))

## 2. Salary-source coverage checks
This is an audit of existing salary inputs and flags, not a live source scrape. Missing or review-flagged salaries stay excluded from the comp-based estimate.

In [ ]:
salaryPairs = pd.read_csv(processedPath / 'salary_cap_normalized_pairs.csv')
salaryCheck = pd.DataFrame({
    'check':['rows','salary available','available salary flagged for review','missing salary flagged for review','salary cap missing','cap share over 100%'],
    'count':[len(salaryPairs),int(salaryPairs.salary_available.fillna(False).sum()),int(salaryPairs.source_review_available_salary.fillna(False).sum()),int(salaryPairs.source_review_missing_salary.fillna(False).sum()),int(salaryPairs.salary_cap_usd.isna().sum()),int(salaryPairs.salary_cap_share_over_100pct.fillna(False).sum())]
})
display(salaryCheck)
print('No salary is used by the profile model or PCA distance.')

## 3. Forecast quality versus persistence
RMSE and MAE are reported per target. RMSE is sensitive to unusually large errors; compare it with MAE and the tail audit below.

In [ ]:
targetMetrics = pd.read_csv(runFolder / 'development_target_metrics.csv')
targetMetrics['rmse_improvement_pct'] = (1 - targetMetrics.model_rmse / targetMetrics.persistence_rmse) * 100
display(targetMetrics.sort_values('rmse_improvement_pct', ascending=False).round(4))
developmentReport = json.loads((runFolder / 'development_report.json').read_text(encoding='utf-8'))
display(pd.Series(developmentReport['profile_pca_metrics'], name='profile metric'))
print('Test evaluated:', developmentReport.get('test_evaluated', 'not recorded'))

## 4. Performance by prior-production tier and tail sensitivity
This runs the audit implementation if its tables are absent, then displays development errors grouped by training-defined prior-PIE tiers.

In [ ]:
tierMetricsPath = runFolder / 'development_metrics_by_prior_pie_tier.csv'
if not tierMetricsPath.exists(): auditProfileTiers()
tierCounts = pd.read_csv(runFolder / 'training_prior_pie_tiers.csv')
tierMetrics = pd.read_csv(tierMetricsPath)
tailMetrics = pd.read_csv(runFolder / 'development_tail_error_audit.csv')
display(tierCounts)
display(tierMetrics[tierMetrics.target.isin(['pie','ws_48','bpm','minutes'])].round(4))
display(tailMetrics.sort_values('model_top_5pct_squared_error_share', ascending=False).round(4))

## 5. Low-game training sensitivity
Compare the unrestricted model with the exploratory model trained after excluding prior seasons below 10 games. This is a sensitivity analysis only; the unrestricted model remains the primary model unless a later decision changes that.

In [ ]:
sensitivityPath = runFolder / 'minimum_games_sensitivity_comparison.csv'
if not sensitivityPath.exists():
    sensitivityFolders = []
    for resultFolder in resultsPath.iterdir():
        metadataFile = modelsPath / resultFolder.name / 'metadata.json'
        if resultFolder.is_dir() and metadataFile.exists():
            modelMetadata = json.loads(metadataFile.read_text(encoding='utf-8'))
            if int(modelMetadata.get('training_minimum_prior_games', 0)) == 10 and (resultFolder / 'minimum_games_sensitivity_comparison.csv').exists():
                sensitivityFolders.append(resultFolder)
    if sensitivityFolders:
        sensitivityPath = max(sensitivityFolders, key=lambda p:p.stat().st_mtime) / 'minimum_games_sensitivity_comparison.csv'
if sensitivityPath.exists():
    sensitivityMetrics = pd.read_csv(sensitivityPath)
    display(sensitivityMetrics[sensitivityMetrics.target.isin(['pie','ws_48','bpm','minutes'])])
else:
    print('Sensitivity artifacts not found. The optional low-game sensitivity workflow has not been migrated into this notebook yet.')

## 6. Nearest-player comps and salary-cap reference
Salary summaries use the median and interquartile range of usable historical comp salaries, converted to the query season's cap. At least five unflagged comp salaries are required. These are descriptive comp estimates, not a salary-trained forecast.

In [ ]:
salaryEstimatePath = runFolder / 'development_comp_salary_estimates.csv'
if not salaryEstimatePath.exists(): estimateSalaryFromComps(runId)
salaryEstimates = pd.read_csv(salaryEstimatePath)
benchEstimates = pd.read_csv(runFolder / 'development_bench_to_starter_salary_estimates.csv')
displayColumns = ['query_player','query_target_season','usable_salary_comp_count','estimated_salary_cap_share_pct','salary_cap_share_q25_pct','salary_cap_share_q75_pct','estimated_salary_usd']
queryRows = salaryEstimates[salaryEstimates.query_player.eq('Isaiah Jackson')]
if queryRows.empty:
    queryRows = salaryEstimates.head(1)
display(queryRows[displayColumns])
display(benchEstimates[displayColumns].head(10))
print('Available estimates: ' + str(salaryEstimates.salary_estimate_available.sum()) + ' / ' + str(len(salaryEstimates)) + ' all-player queries')
print('Available estimates: ' + str(benchEstimates.salary_estimate_available.sum()) + ' / ' + str(len(benchEstimates)) + ' bench-to-starter queries')

neighborPath = runFolder / 'development_bench_to_starter_neighbors.csv'
neighborRows = pd.read_csv(neighborPath)
if (neighborRows.query_player == 'Isaiah Jackson').any():
    display(neighborRows.loc[neighborRows.query_player.eq('Isaiah Jackson'), ['query_player','query_prior_season','query_prior_minutes_per_game','comp_rank','historical_comp','comp_target_minutes_per_game','profile_distance','comp_target_salary_cap_share_pct']].head(10))

## 7. Visual output examples
Forecast accuracy, forecast-vs-observed PCA positions for selected players, and salary-cap shares for Isaiah Jackson’s nearest starter comps. Labels include player names.


In [ ]:
import matplotlib.pyplot as plt

figurePath = runFolder / 'visualizations'
figurePath.mkdir(exist_ok=True)
playerPredictions = pd.read_csv(runFolder / 'development_profile_predictions.csv')
examplePlayers = [('Isaiah Jackson', 2024), ('LeBron James', 2019), ('Stephen Curry', 2014), ('Jayson Tatum', 2024)]

pieFigure, pieAxis = plt.subplots(figsize=(8, 6))
pieAxis.scatter(playerPredictions.actual_pie, playerPredictions.predicted_pie, alpha=0.25, s=18, color='#6688aa')
pieAxis.plot([-0.1, 0.25], [-0.1, 0.25], color='#d1495b', linestyle='--', label='Perfect prediction')
pieAxis.set_xlim(-0.1, 0.25)
pieAxis.set_ylim(-0.1, 0.25)
piePlayers = [('Isaiah Jackson', 2024), ('LeBron James', 2019)]
labelOffsets = {'Isaiah Jackson': (8, -16), 'LeBron James': (8, 12)}
for playerName, seasonYear in piePlayers:
    playerRow = playerPredictions[(playerPredictions.player == playerName) & (playerPredictions.season == seasonYear)]
    if not playerRow.empty:
        playerRow = playerRow.iloc[0]
        pieAxis.annotate(playerName + ' (' + str(seasonYear) + ')', (playerRow.actual_pie, playerRow.predicted_pie), xytext=labelOffsets[playerName], textcoords='offset points', arrowprops={'arrowstyle':'-'}, fontsize=8)
pieAxis.set(title='Development PIE: forecast vs actual (zoomed)', xlabel='Actual next-season PIE', ylabel='Predicted next-season PIE')
pieAxis.legend()
pieFigure.tight_layout()
pieFigure.savefig(figurePath / 'development_pie_forecast.png', dpi=160)
display(pieFigure)

pcaFigure, pcaAxis = plt.subplots(figsize=(9, 7))
pcaAxis.scatter(playerPredictions.actual_pc1, playerPredictions.actual_pc2, alpha=0.18, s=15, label='Observed next-season profiles', color='#999999')
pcaAxis.scatter(playerPredictions.predicted_pc1, playerPredictions.predicted_pc2, alpha=0.18, s=15, label='Forecast profiles', color='#3a86ff')
for playerName, seasonYear in examplePlayers:
    playerRow = playerPredictions[(playerPredictions.player == playerName) & (playerPredictions.season == seasonYear)]
    if not playerRow.empty:
        playerRow = playerRow.iloc[0]
        pcaAxis.plot([playerRow.actual_pc1, playerRow.predicted_pc1], [playerRow.actual_pc2, playerRow.predicted_pc2], color='#d1495b', alpha=0.75, linewidth=1)
        pcaAxis.annotate(playerName, (playerRow.predicted_pc1, playerRow.predicted_pc2), xytext=(5, 5), textcoords='offset points', fontsize=8)
pcaAxis.set(title='PCA skill-profile space: observed vs forecast', xlabel='Principal component 1', ylabel='Principal component 2')
pcaAxis.legend()
pcaFigure.tight_layout()
pcaFigure.savefig(figurePath / 'development_pca_profiles.png', dpi=160)
display(pcaFigure)

isaiahComps = neighborRows[neighborRows.query_player.eq('Isaiah Jackson')].copy()
isaiahComps = isaiahComps[isaiahComps.comp_target_salary_cap_share_pct.notna() & ~isaiahComps.comp_salary_review_flag.fillna(True)].sort_values('comp_rank')
isaiahEstimateRows = benchEstimates[benchEstimates.query_player.eq('Isaiah Jackson')]
if isaiahComps.empty or isaiahEstimateRows.empty: raise ValueError('Isaiah Jackson comps or salary estimate are missing.')
isaiahEstimate = isaiahEstimateRows.iloc[0]
salaryFigure, salaryAxis = plt.subplots(figsize=(10, 6))
salaryAxis.barh(isaiahComps.historical_comp, isaiahComps.comp_target_salary_cap_share_pct, color='#6a994e')
salaryAxis.axvspan(isaiahEstimate.salary_cap_share_q25_pct, isaiahEstimate.salary_cap_share_q75_pct, color='#f2cc8f', alpha=0.35, label='Comp salary IQR')
salaryAxis.axvline(isaiahEstimate.estimated_salary_cap_share_pct, color='#d1495b', linestyle='--', label='Comp median estimate')
salaryAxis.invert_yaxis()
salaryAxis.set(title='Isaiah Jackson: salary shares of nearest starter comps', xlabel='Historical salary as % of salary cap', ylabel='Historical comparable player')
salaryAxis.legend()
salaryFigure.tight_layout()
salaryFigure.savefig(figurePath / 'isaiah_jackson_comp_salary.png', dpi=160)
display(salaryFigure)
print('Saved charts to: ' + str(figurePath))

## Interpretation checklist
- PCA similarity compares predicted skill/efficiency profiles; minutes and salary are excluded from distance.
- Historical comp salary shares are context; the median/IQR is an approximate market reference, not a fair-value guarantee.
- Missing and review-flagged salary records are excluded from estimates.
- Development results are exploratory. The historical test split has been examined previously and must not be presented as fresh independent confirmation.
- No model is trained and no test split is loaded by this notebook.